# CIFAR-10 high-accuracy training (Google Colab)

Is notebook mein ResNet-18, data augmentation, label smoothing, SGD momentum, weight decay aur cosine learning-rate scheduler use hotay hain.

**Colab mein pehle:** `Runtime` → `Change runtime type` → `T4 GPU` select karein. Phir cells ko upar se neeche run karein. Dataset automatically download hoga; local `.tar.gz` upload karne ki zaroorat nahi.

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.optim import SGD
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

# Training configuration
SEED = 42
EPOCHS = 160       # 100 for a quicker run; 160 is recommended for higher accuracy
BATCH_SIZE = 128
LEARNING_RATE = 0.1
MIN_LR = 1e-5
WEIGHT_DECAY = 5e-4
LABEL_SMOOTHING = 0.1
VALIDATION_RATIO = 0.10
WORKERS = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert device.type == 'cuda', 'GPU enable karein: Runtime > Change runtime type > T4 GPU'
print('Using device:', torch.cuda.get_device_name(0))


In [ ]:
# CIFAR-10 normalization values
MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)

# Augmentation sirf training images ke liye use hoti hai.
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.15), ratio=(0.3, 3.3)),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

data_root = Path('/content/cifar10_data')
train_augmented = datasets.CIFAR10(data_root, train=True, transform=train_transform, download=True)
train_for_validation = datasets.CIFAR10(data_root, train=True, transform=eval_transform, download=False)
test_data = datasets.CIFAR10(data_root, train=False, transform=eval_transform, download=True)

validation_size = int(len(train_augmented) * VALIDATION_RATIO)
indices = torch.randperm(len(train_augmented), generator=torch.Generator().manual_seed(SEED)).tolist()
validation_indices = indices[:validation_size]
training_indices = indices[validation_size:]

loader_options = dict(num_workers=WORKERS, pin_memory=True, persistent_workers=True)
train_loader = DataLoader(Subset(train_augmented, training_indices), batch_size=BATCH_SIZE, shuffle=True, **loader_options)
validation_loader = DataLoader(Subset(train_for_validation, validation_indices), batch_size=BATCH_SIZE, shuffle=False, **loader_options)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, **loader_options)

print(f'Train: {len(train_loader.dataset)}, validation: {len(validation_loader.dataset)}, test: {len(test_loader.dataset)}')


In [ ]:
# Standard ResNet-18 ko CIFAR-10 ki 32x32 images ke liye adapt kar rahe hain.
model = models.resnet18(weights=None)
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = SGD(
    model.parameters(), lr=LEARNING_RATE, momentum=0.9,
    weight_decay=WEIGHT_DECAY, nesterov=True
)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=MIN_LR)
scaler = torch.amp.GradScaler('cuda', enabled=True)

def evaluate(loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits = model(images)
                loss = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, 100.0 * correct / total

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')


In [ ]:
# Training: best validation-accuracy checkpoint automatically save hoga.
checkpoint_path = '/content/best_cifar10_resnet18.pt'
best_validation_accuracy = -1.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)

    validation_loss, validation_accuracy = evaluate(validation_loader)
    scheduler.step()
    train_accuracy = 100.0 * correct / total
    print(
        f'Epoch {epoch:03d}/{EPOCHS} | train loss {running_loss / total:.4f} | '
        f'train acc {train_accuracy:.2f}% | val loss {validation_loss:.4f} | '
        f'val acc {validation_accuracy:.2f}% | lr {optimizer.param_groups[0]["lr"]:.6f}'
    )

    if validation_accuracy > best_validation_accuracy:
        best_validation_accuracy = validation_accuracy
        torch.save({
            'epoch': epoch,
            'validation_accuracy': validation_accuracy,
            'model_state_dict': model.state_dict(),
            'class_names': train_augmented.classes,
        }, checkpoint_path)
        print('  Best model saved:', checkpoint_path)


In [ ]:
# Best checkpoint ko load kar ke final TEST accuracy nikalain.
best_checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(best_checkpoint['model_state_dict'])
test_loss, test_accuracy = evaluate(test_loader)

print('\nTraining complete')
print(f"Best validation accuracy: {best_checkpoint['validation_accuracy']:.2f}% (epoch {best_checkpoint['epoch']})")
print(f'Final test loss: {test_loss:.4f}')
print(f'Final test accuracy: {test_accuracy:.2f}%')


In [ ]:
# Optional: trained model ko apne computer par download karne ke liye is cell ko run karein.
from google.colab import files
files.download(checkpoint_path)
